# 03 - Transformer with Defense 1 (Regularization)

Defense: L2 + Dropout + EarlyStopping

In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping

from research_protocol import (
    evaluate_mia_research_protocol,
    compare_defense_vs_baseline,
    set_seed,
)
from research_protocol_robust import evaluate_mia_robust_protocol

In [2]:
ATTACK_SEEDS = [11, 22]

# Attaquant fort (meme base que notebook 02)
N_SHADOWS = 4
SHADOW_EPOCHS = 4
SHADOW_BATCH = 16

ROBUST_N_SHADOWS = 6
ROBUST_SHADOW_EPOCHS = 4
ROBUST_BOUNDARY_DIRS = 8
ROBUST_BOUNDARY_STEPS = 4
ROBUST_BOUNDARY_MAX_ALPHA = 2.0


def build_transformer_base(n_features, dropout=0.15):
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(64)(inp)
    a = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(128, activation='relu')(x)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(64)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m


def build_transformer_reg(n_features, dropout=0.35, l2v=1e-4):
    reg = regularizers.l2(l2v)
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(64, kernel_regularizer=reg)(inp)
    a = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(128, activation='relu', kernel_regularizer=reg)(x)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(64, kernel_regularizer=reg)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m

In [3]:
# Model override: DistilBERT adapter with keras-like API
from model_adapters import make_standard_model

def build_transformer_base(n_features, dropout=0.15):
    _ = n_features
    return make_standard_model(dropout=dropout)

def build_transformer_reg(n_features, dropout=0.35, l2v=1e-4):
    _ = n_features
    return make_standard_model(dropout=dropout, l2v=l2v)

c:\Users\khalil\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print('=== Load prepared data ===')
b = np.load('../data_preparation/prepared_oasis2_transformer.npz', allow_pickle=True)
X, y = b['X'], b['y']
X_train, X_test = b['X_train'], b['X_test']
y_train, y_test = b['y_train'], b['y_test']
X_train_seq, X_test_seq = b['X_train_seq'], b['X_test_seq']
X_shadow_raw, y_shadow = b['X_shadow_raw'], b['y_shadow']
SEED = int(b['seed'][0])
print(f'X={X.shape}, train={X_train.shape}, test={X_test.shape}, shadow={X_shadow_raw.shape}')

=== Load prepared data ===
X=(354, 9), train=(70, 9), test=(284, 9), shadow=(107, 9)


In [5]:
print('=== Train Defense 1 model ===')
set_seed(SEED + 10); tf.keras.backend.clear_session()
model_def1 = build_transformer_reg(X_train.shape[1], dropout=0.35, l2v=1e-4)
es = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0)
h = model_def1.fit(X_train_seq, y_train, epochs=12, batch_size=32, validation_split=0.2, callbacks=[es], verbose=0)
p_te = model_def1.predict(X_test_seq, verbose=0).ravel()
test_acc_def1 = accuracy_score(y_test, (p_te >= 0.5).astype(int))
print(f'test_acc_def1={test_acc_def1:.4f}, epochs={len(h.history["loss"])}')

=== Train Defense 1 model ===



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


test_acc_def1=0.7570, epochs=12


In [6]:
print('=== Run MIA attacks with unified research protocol (strong attacker) ===')

print('Baseline attack-target training (risky)...')
set_seed(SEED + 1)
tf.keras.backend.clear_session()
model_base = build_transformer_base(X_train.shape[1], dropout=0.0)
model_base.fit(X_train_seq, y_train, epochs=10, batch_size=32, verbose=0)
baseline_test_acc = ((model_base.predict(X_test_seq, verbose=0).ravel() >= 0.5).astype(int) == y_test).mean()

baseline_summary, baseline_per_seed = evaluate_mia_research_protocol(
    target_model=model_base,
    X_train_seq=X_train_seq,
    y_train=y_train,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer_base(nf, dropout=0.0),
    seed_list=ATTACK_SEEDS,
    n_shadows=N_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
)

print('Defense 1 target training...')
def1_summary, def1_per_seed = evaluate_mia_research_protocol(
    target_model=model_def1,
    X_train_seq=X_train_seq,
    y_train=y_train,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer_reg(nf, dropout=0.35, l2v=1e-4),
    seed_list=ATTACK_SEEDS,
    n_shadows=N_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
)

print('Baseline summary (mean Â± std):')
display(baseline_summary.round(4))
print('Defense 1 summary (mean Â± std):')
display(def1_summary.round(4))

=== Run MIA attacks with unified research protocol (strong attacker) ===
Baseline attack-target training (risky)...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-strea

Defense 1 target training...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-strea

Baseline summary (mean Â± std):


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
1,shadow_meta,0.5414,0.0369,0.5387,0.0050,0.2135,0.0147,0.5000,0.0505,0.2992,0.0235
0,logistic,0.5158,0.0113,0.8028,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,threshold_loss,0.5158,0.0113,0.5070,0.2091,0.2030,0.0393,0.4643,0.2020,0.2703,0.0024


Defense 1 summary (mean Â± std):


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,logistic,0.5517,0.0071,0.8028,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,threshold_loss,0.5517,0.0071,0.6021,0.0149,0.2383,0.0043,0.4643,0.0505,0.3147,0.0154
1,shadow_meta,0.4623,0.0008,0.3521,0.0199,0.1980,0.0028,0.7500,0.0505,0.3133,0.0079


In [7]:
print('=== Baseline vs Defense 1 (unified protocol) ===')
cmp = compare_defense_vs_baseline(baseline_summary, def1_summary, 'defense1')
display(cmp.round(4))

mcmp = pd.DataFrame([
    {
        'model': 'Transformer',
        'test_acc_baseline': float(baseline_test_acc),
        'test_acc_defense1': float(test_acc_def1),
        'delta_test_acc': float(test_acc_def1 - baseline_test_acc),
    }
])
display(mcmp.round(4))

=== Baseline vs Defense 1 (unified protocol) ===


,attack,auc_mean_baseline,accuracy_mean_baseline,f1_mean_baseline,auc_mean_defense1,accuracy_mean_defense1,f1_mean_defense1,delta_auc,delta_accuracy,delta_f1
1,logistic,0.5158,0.8028,0.0000,0.5517,0.8028,0.0000,0.0359,0.0000,0.0000
0,shadow_meta,0.5414,0.5387,0.2992,0.4623,0.3521,0.3133,-0.0791,-0.1866,0.0141
2,threshold_loss,0.5158,0.5070,0.2703,0.5517,0.6021,0.3147,0.0359,0.0951,0.0444


,model,test_acc_baseline,test_acc_defense1,delta_test_acc
0,Transformer,0.919,0.757,-0.162


In [8]:
print('=== Quick AUC summary ===')
quick_auc = cmp[['attack', 'auc_mean_baseline', 'auc_mean_defense1', 'delta_auc']].sort_values('delta_auc')
display(quick_auc.round(4))

=== Quick AUC summary ===


,attack,auc_mean_baseline,auc_mean_defense1,delta_auc
0,shadow_meta,0.5414,0.4623,-0.0791
1,logistic,0.5158,0.5517,0.0359
2,threshold_loss,0.5158,0.5517,0.0359


In [9]:
print('=== Robust adaptive attack (meta + LiRA + boundary): baseline vs defense1 ===')

base_robust_summary, base_robust_per_seed = evaluate_mia_robust_protocol(
    target_model=model_base,
    x_train_seq=X_train_seq,
    y_train=y_train,
    x_test_seq=X_test_seq,
    y_test=y_test,
    x_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer_base(nf, dropout=0.0),
    seed_list=ATTACK_SEEDS,
    n_shadows=ROBUST_N_SHADOWS,
    shadow_epochs=ROBUST_SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
    boundary_dirs=ROBUST_BOUNDARY_DIRS,
    boundary_steps=ROBUST_BOUNDARY_STEPS,
    boundary_max_alpha=ROBUST_BOUNDARY_MAX_ALPHA,
)

def1_robust_summary, def1_robust_per_seed = evaluate_mia_robust_protocol(
    target_model=model_def1,
    x_train_seq=X_train_seq,
    y_train=y_train,
    x_test_seq=X_test_seq,
    y_test=y_test,
    x_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer_reg(nf, dropout=0.35, l2v=1e-4),
    seed_list=ATTACK_SEEDS,
    n_shadows=ROBUST_N_SHADOWS,
    shadow_epochs=ROBUST_SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
    boundary_dirs=ROBUST_BOUNDARY_DIRS,
    boundary_steps=ROBUST_BOUNDARY_STEPS,
    boundary_max_alpha=ROBUST_BOUNDARY_MAX_ALPHA,
)

cmp_robust = compare_defense_vs_baseline(base_robust_summary, def1_robust_summary, 'defense1')
print('=== Quick AUC summary (robust adaptive) ===')
quick_auc_robust = cmp_robust[['attack', 'auc_mean_baseline', 'auc_mean_defense1', 'delta_auc']].sort_values('delta_auc')
display(quick_auc_robust.round(4))

=== Robust adaptive attack (meta + LiRA + boundary): baseline vs defense1 ===


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-strea

=== Quick AUC summary (robust adaptive) ===


,attack,auc_mean_baseline,auc_mean_defense1,delta_auc
0,shadow_meta,0.5315,0.5437,0.0122
